### Modelo Optimización Asignación Box (Gurobi)

In [3]:
import sys
import re
import unicodedata
from math import gcd
from functools import reduce
from pathlib import Path

import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl import comments as xl_comments


# ============================================================
# UTILIDAD: normalización de texto
# ============================================================

def normalize(text: str) -> str:
    return ''.join(
        c for c in unicodedata.normalize('NFD', str(text).upper().strip())
        if unicodedata.category(c) != 'Mn'
    )


# ============================================================
# RUTAS DE ARCHIVOS
# ============================================================

EXCEL_AGENDAS  = "Agendas_Hospital_Grant.xlsx"
EXCEL_MAESTRO  = "Excel_Maestro_Boxes.xlsx"

for path in [EXCEL_AGENDAS, EXCEL_MAESTRO]:
    if not Path(path).is_file():
        sys.exit(f"No se encontró el archivo: '{path}'")

print(f"Agendas : {EXCEL_AGENDAS}")
print(f"Maestro : {EXCEL_MAESTRO}")


# ============================================================
# SECCIÓN 1 – PARÁMETROS GLOBALES FIJOS
# ============================================================

START_MIN = 8 * 60
END_MIN   = 17 * 60
DAYS      = ["LUNES", "MARTES", "MIERCOLES", "JUEVES", "VIERNES"]

W1 = 100   # penalización actividad sin box
W2 = 40   # penalización cambio de box

ACTIVITY_CODES = {
    "CONSULTA EXAMEN FISICO":     "CEF",
    "CONSULTA SIN EXAMEN FISICO": "CSF",
    "PROCEDIMIENTO":              "PROC",
}
A     = list(set(ACTIVITY_CODES.values()))
DELTA = reduce(gcd, [30, 30, 30])   # minutos por bloque
T     = list(range(1, (END_MIN - START_MIN) // DELTA + 1))


def time_to_block(time_str) -> int:
    from datetime import time as dt_time, datetime as dt_datetime
    if isinstance(time_str, dt_time):
        h, m = time_str.hour, time_str.minute
    elif isinstance(time_str, dt_datetime):
        h, m = time_str.hour, time_str.minute
    else:
        parts = str(time_str).strip().split(":")
        h, m = int(parts[0]), int(parts[1])
    return (h * 60 + m - START_MIN) // DELTA + 1


def block_to_time(t: int) -> str:
    mins = START_MIN + (t - 1) * DELTA
    return f"{mins // 60:02d}:{mins % 60:02d}"


# ============================================================
# SECCIÓN 2 – CARGA DEL EXCEL MAESTRO
# Construye dinámicamente: B, cap_b, COMBA, POLI, B_poli,
# BoxPoli, HAB_BOX_NM, HAB_POLI_NM, REGLAS
# ============================================================

def load_maestro(filepath: str) -> dict:
    xl = pd.read_excel(filepath, sheet_name=None, header=None)

    # ── Hoja Boxes ──────────────────────────────────────────
    df_b = pd.read_excel(filepath, sheet_name="Boxes", header=2)
    df_b.columns = [str(c).strip() for c in df_b.columns]
    df_b = df_b[df_b["BOX_ID"].notna()]
    df_b = df_b[df_b["BOX_ID"].astype(str).str.startswith("BOX")]

    box_ids  = []
    cap_b    = {}
    labels_b = {}
    comba    = {}

    for _, row in df_b.iterrows():
        bid           = str(row["BOX_ID"]).strip()
        cap_b[bid]    = int(row.get("cap_b", 1) or 1)
        labels_b[bid] = str(row.get("LABEL", bid)).strip()
        # Buscar columnas con saltos de línea en el encabezado
        col_cef  = next((c for c in df_b.columns if c.startswith("CEF")),  "CEF")
        col_csf  = next((c for c in df_b.columns if c.startswith("CSF")),  "CSF")
        col_proc = next((c for c in df_b.columns if c.startswith("PROC")), "PROC")
        comba[(bid, "CEF")]  = int(row.get(col_cef,  0) or 0)
        comba[(bid, "CSF")]  = int(row.get(col_csf,  0) or 0)
        comba[(bid, "PROC")] = int(row.get(col_proc, 0) or 0)
        box_ids.append(bid)

    # Completar COMBA para actividades no listadas
    for b in box_ids:
        for a in A:
            if (b, a) not in comba:
                comba[(b, a)] = 0

    # ── Hoja Policlinicos ────────────────────────────────────
    df_p = pd.read_excel(filepath, sheet_name="Policlinicos", header=2)
    df_p.columns = [str(c).strip() for c in df_p.columns]
    df_p = df_p[df_p["POLICLINICO_ID"].notna()]
    df_p = df_p[~df_p["POLICLINICO_ID"].astype(str).str.startswith("nan")]

    poli_ids      = []
    b_poli        = {}
    # Mapa nombre_archivo (sin extensión, normalizado) → poli_id
    archivo_to_poli = {}

    for _, row in df_p.iterrows():
        pid      = str(row["POLICLINICO_ID"]).strip()
        bxs      = [b.strip() for b in str(row.get("BOXES (separados por coma)", "")).split(",") if b.strip()]
        nom_arch = str(row.get("NOMBRE_ARCHIVO\n(sin extensión)", 
                      row.get("NOMBRE_ARCHIVO", ""))).strip()
        poli_ids.append(pid)
        b_poli[pid]        = bxs
        if nom_arch and nom_arch != "nan":
            archivo_to_poli[normalize(nom_arch)] = pid

    # BoxPoli: registro de referencia box → primer policlínico encontrado.
    # OJO: un box puede pertenecer a VARIOS policlínicos (muchos-a-muchos).
    # Este diccionario ya NO se usa para construir BoxPoli_param (eso se
    # hace directamente desde b_poli más abajo, en la Sección 2, para no
    # perder los policlínicos adicionales). Se mantiene aquí solo como
    # referencia informativa y para detectar/avisar boxes compartidos.
    box_poli_map = {}
    boxes_compartidos = []
    for pid, bxs in b_poli.items():
        for b in bxs:
            if b in box_poli_map and box_poli_map[b] != pid:
                boxes_compartidos.append((b, box_poli_map[b], pid))
            box_poli_map.setdefault(b, pid)

    if boxes_compartidos:
        print(f"\n  [INFO] Boxes compartidos entre policlínicos "
              f"(esto es válido, BoxPoli_param los habilita en ambos): "
              f"{boxes_compartidos}")

    # ── Hoja Habilitaciones_NM ───────────────────────────────
    df_nm = pd.read_excel(filepath, sheet_name="Habilitaciones_NM", header=2)
    df_nm.columns = [str(c).strip() for c in df_nm.columns]
    df_nm = df_nm[df_nm["PROFESION_CANON"].notna()]
    df_nm = df_nm[~df_nm["PROFESION_CANON"].astype(str).str.startswith("nan")]

    hab_box_nm    = {}   # {canon: [box_ids]}
    hab_poli_nm   = {}   # {canon: [poli_ids]}
    kw_to_canon   = {}   # {keyword: canon}  para detección
    profesiones_nm_set = set()

    for _, row in df_nm.iterrows():
        canon  = str(row["PROFESION_CANON"]).strip()
        kws    = [k.strip() for k in str(row.get("KEYWORDS_DETECCION","")).split(",") if k.strip()]
        bxs    = [b.strip() for b in str(row.get("BOXES_HABILITADOS","")).split(",") if b.strip()]
        pols   = [p.strip() for p in str(row.get("POLICLINICOS_HABILITADOS","")).split(",") if p.strip()]
        hab_box_nm[canon]  = bxs
        hab_poli_nm[canon] = pols
        profesiones_nm_set.add(canon)
        for kw in kws:
            kw_to_canon[normalize(kw)] = canon

    # ── Hoja Especialidades_Poli (EspPoli: especialidad médica → policlínicos) ──
    df_e = pd.read_excel(filepath, sheet_name="Especialidades_Poli", header=2)
    df_e.columns = [str(c).strip() for c in df_e.columns]
    df_e = df_e[df_e["ESPECIALIDAD_CANON"].notna()]
    df_e = df_e[~df_e["ESPECIALIDAD_CANON"].astype(str).str.startswith("nan")]

    esp_poli = {}        # {(especialidad_canon, poli_id): 1}
    especialidades_canon = set()
    for _, row in df_e.iterrows():
        esp_canon = normalize(str(row["ESPECIALIDAD_CANON"]).strip())
        pols = [p.strip() for p in str(row.get("POLICLINICOS_HABILITADOS", "")).split(",") if p.strip()]
        especialidades_canon.add(esp_canon)
        for pid_poli in pols:
            esp_poli[(esp_canon, pid_poli)] = 1

    # ── Hoja Reglas_Especiales ───────────────────────────────
    df_r = pd.read_excel(filepath, sheet_name="Reglas_Especiales", header=2)
    df_r.columns = [str(c).strip() for c in df_r.columns]
    TIPOS_VALIDOS = {"BOX_EXCLUSIVO", "BOX_FORZADO"}
    df_r = df_r[df_r["TIPO"].isin(TIPOS_VALIDOS)]

    reglas = []
    for _, row in df_r.iterrows():
        rid   = str(row.get("ID","")).strip()
        tipo  = str(row.get("TIPO","")).strip()
        prof  = normalize(str(row.get("PROFESIONAL","")).strip())
        dia   = str(row.get("DIA","")).strip().upper()
        hd    = str(row.get("HORA_DESDE","")).strip()
        hh    = str(row.get("HORA_HASTA","")).strip()
        act   = normalize(str(row.get("ACTIVIDAD_NORM","")).strip())
        box   = str(row.get("BOX_FORZADO","")).strip()
        if rid and tipo and prof:
            try:
                t_from = time_to_block(hd)
                t_to   = time_to_block(hh)
                blocks = set(range(t_from, t_to))
            except Exception:
                blocks = set()
            reglas.append({
                "id": rid, "tipo": tipo, "prof_norm": prof,
                "dia": dia, "blocks": blocks,
                "act_norm": act, "box": box,
            })

    print(f"\nMaestro cargado:")
    print(f"  Boxes          : {len(box_ids)}")
    print(f"  Policlínicos   : {len(poli_ids)}  → {poli_ids}")
    print(f"  Especialidades : {len(especialidades_canon)} → {sorted(especialidades_canon)}")
    print(f"  Profesiones NM : {len(profesiones_nm_set)} → {sorted(profesiones_nm_set)}")
    print(f"  Reglas especiales: {len(reglas)}")

    return {
        "box_ids": box_ids, "cap_b": cap_b, "labels": labels_b,
        "comba": comba,
        "poli_ids": poli_ids, "b_poli": b_poli,
        "box_poli_map": box_poli_map,
        "hab_box_nm": hab_box_nm, "hab_poli_nm": hab_poli_nm,
        "kw_to_canon": kw_to_canon, "profesiones_nm_set": profesiones_nm_set,
        "esp_poli": esp_poli, "especialidades_canon": especialidades_canon,
        "reglas": reglas,
        "archivo_to_poli": archivo_to_poli,
    }


print("\n" + "=" * 60)
print("CARGANDO EXCEL MAESTRO")
print("=" * 60)
M = load_maestro(EXCEL_MAESTRO)

# ── Detectar policlínico: primero desde nombre de archivo, sino desde Maestro ──
_nombre_arch_norm = normalize(Path(EXCEL_AGENDAS).stem)
POLI_AGENDAS = M["archivo_to_poli"].get(_nombre_arch_norm)
if POLI_AGENDAS is None:
    print(f"\n  [AVISO] '{Path(EXCEL_AGENDAS).stem}' no encontrado en Maestro. "
          f"Se leerá desde el encabezado de cada hoja de agenda.")
else:
    print(f"\nPoliclínico detectado desde nombre de archivo: '{POLI_AGENDAS}'")

# Parámetros estructurales derivados del Maestro
B            = M["box_ids"]
cap_b        = M["cap_b"]
COMBA        = M["comba"]
POLI         = M["poli_ids"]
B_poli       = M["b_poli"]
box_poli_map = M["box_poli_map"]
REGLAS       = M["reglas"]

# Boxes exclusivos de NM = todos los boxes habilitados para alguna profesión NM
BOXES_EXCLUSIVOS_NM = set(
    b for bxs in M["hab_box_nm"].values() for b in bxs
)
# Boxes de reglas BOX_EXCLUSIVO (ej BOX_CORSET) también son exclusivos
BOXES_EXCLUSIVOS_REGLA = {r["box"] for r in REGLAS if r["tipo"] == "BOX_EXCLUSIVO"}

# BoxPoli params — construido DIRECTAMENTE desde B_poli (policlínico ->
# lista de boxes), que ya es correctamente muchos-a-muchos: un mismo box
# puede tener BoxPoli_param[(b,rho)] = 1 para más de un policlínico rho.
# (Antes se derivaba de box_poli_map, que solo guardaba un policlínico
# por box y perdía los policlínicos adicionales cuando un box era
# compartido — ver aviso [INFO] impreso en load_maestro.)
BoxPoli_param = {(b, rho): 0 for b in B for rho in POLI}
for rho_id, bxs in B_poli.items():
    for b in bxs:
        if b in B:
            BoxPoli_param[(b, rho_id)] = 1


# ============================================================
# SECCIÓN 3 – LECTURA DE AGENDAS Y CONSTRUCCIÓN DSA_M / DSA_NM
# ============================================================

def detect_nm_canon(profesion_norm: str, kw_to_canon: dict) -> str | None:
    """Detecta si una profesión normalizada es no médica y retorna su canon."""
    for kw, canon in kw_to_canon.items():
        if kw in profesion_norm:
            return canon
    return None


def parse_agendas(filepath: str, maestro: dict, poli_agendas: str):
    xl = pd.read_excel(filepath, sheet_name=None, header=None)

    professionals = []
    DSA_M  = {}
    DSA_NM = {}
    skipped = []

    kw_to_canon = maestro["kw_to_canon"]

    KEYWORD_CANON_MEDICO = {
        "CIRUJANO INFANTIL":        "CIRUJANO INFANTIL",
        "TRAUMATOLOGO INFANTIL":    "TRAUMATOLOGO INFANTIL",
        "TRAUMATOLOG":              "TRAUMATOLOGO INFANTIL",
        "ODONTOLOGO":               "ODONTOLOGO",
        "DENTAL":                   "ODONTOLOGO",
        "CIRUJANO MAXILO":          "CIRUJANO MAXILO-FACIAL",
        "MAXILO":                   "CIRUJANO MAXILO-FACIAL",
        "CIRUGIA INFANTIL Y ORTOP": "CIRUGIA INFANTIL Y ORTOPEDIA",
        "CIRUGIA INFANTIL":         "CIRUJANO INFANTIL",
    }

    for sheet_name, df in xl.items():
        name       = None
        profesion  = None
        subesp     = None
        amb_hours  = None
        policlinico = poli_agendas

        for i in range(min(20, len(df))):
            for j in range(df.shape[1]):
                vj = str(df.iloc[i, j]) if not pd.isna(df.iloc[i, j]) else ""

                if "POLICLINICO:" in normalize(vj) and policlinico == poli_agendas:
                    raw_poli = normalize(vj).split("POLICLINICO:")[-1].strip()
                    matched_poli = next(
                        (pid for pid in maestro["poli_ids"]
                         if normalize(pid) == raw_poli or raw_poli in normalize(pid)),
                        None
                    )
                    if matched_poli:
                        policlinico = matched_poli

            v0 = str(df.iloc[i, 0]) if not pd.isna(df.iloc[i, 0]) else ""
            if "NOMBRE PROFESIONAL:" in v0:
                name = v0.replace("NOMBRE PROFESIONAL:", "").strip()
            if "PROFESION:" in v0.upper() and profesion is None:
                profesion = normalize(v0.upper().replace("PROFESION:", "").strip())
            if "SUBESPECIALIDAD:" in v0 and subesp is None:
                subesp = normalize(v0.replace("SUBESPECIALIDAD:", "").strip())
            for j in range(df.shape[1]):
                vj = str(df.iloc[i, j]) if not pd.isna(df.iloc[i, j]) else ""
                if "HORAS DE ATENCION AMBULATORIA" in vj:
                    nums = re.findall(r"[\d,\.]+", vj)
                    if nums:
                        try:
                            amb_hours = float(nums[-1].replace(",", "."))
                        except ValueError:
                            pass

        if name is None:
            skipped.append((sheet_name, "sin NOMBRE PROFESIONAL"))
            continue

        prof_norm  = profesion if profesion else ""
        nm_canon   = detect_nm_canon(prof_norm, kw_to_canon)
        is_nm      = nm_canon is not None

        if is_nm:
            specialty = nm_canon
        else:
            matched = None
            if subesp:
                for kw, canon in KEYWORD_CANON_MEDICO.items():
                    if kw in subesp:
                        matched = canon
                        break
            specialty = matched if matched else "CIRUJANO INFANTIL"
            if matched is None and subesp:
                print(f"  [{sheet_name}] subesp '{subesp}' no reconocida → 'CIRUJANO INFANTIL'")

        palt    = 1 if (amb_hours is not None and amb_hours > 22) else 0
        prof_id = f"P{len(professionals)+1}_{name.replace(' ','_')}"

        professionals.append({
            "id":          prof_id,
            "name":        name,
            "name_norm":   normalize(name),
            "specialty":   specialty,
            "profesion":   prof_norm,
            "nm_canon":    nm_canon,
            "is_nm":       is_nm,
            "policlinico": policlinico,
            "amb_hours":   amb_hours,
            "palt":        palt,
        })

        header_row = None
        for i, row in df.iterrows():
            if "DIA" in str(row.iloc[0]) and "ACTIVIDAD" in str(row.iloc[1]):
                header_row = i
                break
        if header_row is None:
            skipped.append((sheet_name, "sin fila DIA/ACTIVIDAD"))
            continue

        current_day = None
        for i in range(header_row + 2, len(df)):
            row       = df.iloc[i]
            dia       = str(row.iloc[0]).strip()
            act_name  = str(row.iloc[1]).strip()
            tipo      = str(row.iloc[2]).strip()
            desde_raw = row.iloc[3]
            hasta_raw = row.iloc[4]
            desde     = str(desde_raw).strip() if desde_raw is not None else "nan"
            hasta     = str(hasta_raw).strip() if hasta_raw is not None else "nan"

            if dia not in ("nan", "NaN", ""):
                current_day = dia.upper().strip()

            if "NOMBRE" in act_name or "FIRMA" in act_name:
                break
            if act_name in ("nan","NaN","") or tipo in ("nan","NaN",""):
                continue
            if current_day not in DAYS:
                continue
            if desde in ("nan","NaN","None","") or hasta in ("nan","NaN","None",""):
                continue

            tipo_up = normalize(tipo)
            if "SIN EXAMEN" in tipo_up:      act_code = "CSF"
            elif "EXAMEN" in tipo_up:        act_code = "CEF"
            elif "PROCEDIMIENTO" in tipo_up: act_code = "PROC"
            else:
                continue

            try:
                b_from = time_to_block(desde_raw)
                b_to   = time_to_block(hasta_raw)
            except Exception:
                continue

            for t in range(b_from, b_to):
                if t in T:
                    if is_nm:
                        DSA_NM[(prof_id, act_code, current_day, t)] = 1
                    else:
                        DSA_M[(prof_id, act_code, specialty, current_day, t)] = 1

    if skipped:
        print(f"\n  Hojas omitidas ({len(skipped)}):")
        for sn, reason in skipped:
            print(f"    · '{sn}': {reason}")

    return professionals, DSA_M, DSA_NM


print("\n" + "=" * 60)
print("LEYENDO AGENDAS")
print("=" * 60)
professionals, DSA_M, DSA_NM = parse_agendas(EXCEL_AGENDAS, M, POLI_AGENDAS)

P_M  = [p["id"] for p in professionals if not p["is_nm"]]
P_NM = [p["id"] for p in professionals if     p["is_nm"]]
P    = P_M + P_NM

name_to_pid = {p["name_norm"]: p["id"] for p in professionals}

E = list(set(p["specialty"] for p in professionals if not p["is_nm"]))

EspPoli_param = {
    (e, rho): M["esp_poli"].get((normalize(e), rho), 0)
    for e in E for rho in POLI
}

especialidades_sin_registrar = [
    e for e in E if normalize(e) not in M["especialidades_canon"]
]
if especialidades_sin_registrar:
    print(f"  [AVISO] Especialidades sin registrar en 'Especialidades_Poli': "
          f"{especialidades_sin_registrar} → no podrán ser asignadas a ningún box "
          f"hasta que se agreguen a esa hoja.")

especialidades_sin_poli = [
    e for e in E
    if normalize(e) in M["especialidades_canon"]
    and not any(EspPoli_param.get((e, rho), 0) == 1 for rho in POLI)
]
if especialidades_sin_poli:
    print(f"  [AVISO] Especialidades registradas pero SIN policlínico habilitado: "
          f"{especialidades_sin_poli} → no podrán ser asignadas a ningún box "
          f"hasta que se complete la columna POLICLINICOS_HABILITADOS.")

Palt = {p["id"]: p["palt"] for p in professionals}

EspP = {
    (p["id"], e): (1 if p["specialty"] == e else 0)
    for p in professionals if not p["is_nm"]
    for e in E
}

HabEspPoli = {}
for p in professionals:
    pid = p["id"]
    if p["is_nm"]:
        for b in B:
            HabEspPoli[(pid, b)] = 1
        continue
    especialidades_p = [e for e in E if EspP.get((pid, e), 0) == 1]
    for b in B:
        # Revisa TODOS los policlínicos a los que pertenece el box (un
        # box compartido puede estar en más de uno), no solo el primero.
        rhos_box = [rho for rho in POLI if BoxPoli_param.get((b, rho), 0) == 1]
        compatible = any(
            EspPoli_param.get((e, rho), 0) == 1
            for e in especialidades_p for rho in rhos_box
        )
        HabEspPoli[(pid, b)] = 1 if compatible else 0

HabBox = {}
for p in professionals:
    pid = p["id"]
    if p["is_nm"]:
        hab = set(M["hab_box_nm"].get(p["nm_canon"], []))
        for b in B:
            HabBox[(pid, b)] = 1 if b in hab else 0
    else:
        boxes_regla_excl = {
            r["box"] for r in REGLAS
            if r["tipo"] == "BOX_EXCLUSIVO" and r["prof_norm"] == p["name_norm"]
        }
        for b in B:
            if b in BOXES_EXCLUSIVOS_NM:
                HabBox[(pid, b)] = 0
            elif b in BOXES_EXCLUSIVOS_REGLA:
                HabBox[(pid, b)] = 1 if b in boxes_regla_excl else 0
            else:
                HabBox[(pid, b)] = 1

HabPoli = {}
for p in professionals:
    pid = p["id"]
    for rho in POLI:
        if p["is_nm"]:
            hab_polis = M["hab_poli_nm"].get(p["nm_canon"], [])
            HabPoli[(pid, rho)] = 1 if rho in hab_polis else 0
        else:
            HabPoli[(pid, rho)] = 1 if rho == p["policlinico"] else 0

print(f"\nDelta = {DELTA} min  |T| = {len(T)} bloques/día")
print(f"Especialidades detectadas: {E}")
print(f"\nP_M  ({len(P_M)} médicos):")
for p in professionals:
    if not p["is_nm"]:
        print(f"  {p['id'][:30]:30s} | {p['specialty']:30s} | "
              f"poli={p['policlinico']} palt={p['palt']}")
print(f"\nP_NM ({len(P_NM)} no médicos):")
for p in professionals:
    if p["is_nm"]:
        hab_b = M["hab_box_nm"].get(p["nm_canon"], [])
        hab_r = M["hab_poli_nm"].get(p["nm_canon"], [])
        print(f"  {p['id'][:30]:30s} | {p['nm_canon']:18s} | HabBox={hab_b} HabPoli={hab_r}")
print(f"\nDSA_M={len(DSA_M)}  DSA_NM={len(DSA_NM)}")
print(f"Reglas especiales: {[r['id'] for r in REGLAS]}")


# ============================================================
# SECCIÓN 4 – MODELO GUROBI
# ============================================================

print("\n" + "=" * 60)
print("CONSTRUYENDO MODELO GUROBI")
print("=" * 60)

model = gp.Model("AsignacionBoxes_v6_General")
model.setParam("TimeLimit", 300)
model.setParam("OutputFlag", 1)

pid_to_prof = {p["id"]: p for p in professionals}

regla_pid_map = []
for r in REGLAS:
    pid = name_to_pid.get(r["prof_norm"])
    if pid:
        regla_pid_map.append({**r, "pid": pid})
    else:
        print(f"  [AVISO] Regla {r['id']}: profesional '{r['prof_norm']}' no encontrado en agendas.")

def get_regla_box(pid, a_code, day, t):
    act_n = a_code
    for r in regla_pid_map:
        if (r["pid"] == pid and r["tipo"] == "BOX_FORZADO"
                and (r["dia"] == day or r["dia"] == "*")
                and t in r["blocks"]):
            return r["box"]
    return None

def is_box_exclusivo_para(b, pid, a_code, day, t):
    for r in regla_pid_map:
        if (r["box"] == b and r["tipo"] == "BOX_EXCLUSIVO"
                and r["pid"] == pid
                and (r["dia"] == day or r["dia"] == "*")
                and t in r["blocks"]
                and COMBA.get((b, a_code), 0) == 1):
            return True
    return False

# ─────────────────────────────────────────────────────────────────
#  PARÁMETROS DE REGLAS ESPECIALES (para restricciones 8.11 y 8.12)
# ─────────────────────────────────────────────────────────────────
# Forz[(p,b,d,t)] = 1 : el profesional p DEBE ubicarse en el box b en el
#                       día d, bloque t (BOX_FORZADO + parte forzada de
#                       BOX_EXCLUSIVO).
# Excl[(p,b)]     = 1 : el box b es de uso EXCLUSIVO del profesional p.
# B_EX            : conjunto de boxes exclusivos (B^EX en la formulación).
Forz = {}
Excl = {}
B_EX = set()
for r in regla_pid_map:
    pid = r["pid"]
    box = r["box"]
    dias_r = DAYS if r["dia"] == "*" else [r["dia"]]
    for d in dias_r:
        for t in r["blocks"]:
            Forz[(pid, box, d, t)] = 1
    if r["tipo"] == "BOX_EXCLUSIVO":
        Excl[(pid, box)] = 1
        B_EX.add(box)


def demanda(pid, a, d, t):
    """DSA combinada (médica o no médica) del profesional en ese slot."""
    prof = pid_to_prof[pid]
    if prof["is_nm"]:
        return DSA_NM.get((pid, a, d, t), 0)
    return sum(DSA_M.get((pid, a, e, d, t), 0) for e in E)


# ── VARIABLES x : se crean sobre TODOS los boxes ──────────────────────
# La factibilidad la deciden las restricciones 8.5-8.9 y 8.11-8.12
# (todas explícitas más abajo), no un pre-filtrado silencioso.
x = {}
todas_dsa = list(set(
    [(p, a, d, t) for (p, a, e, d, t) in DSA_M] +
    [(p, a, d, t) for (p, a, d, t)    in DSA_NM]
))
for (pid, a, d, t) in todas_dsa:
    for b in B:
        x[pid, a, b, d, t] = model.addVar(
            vtype=GRB.BINARY, name=f"x_{pid}_{a}_{b}_{d}_{t}")

# ── VARIABLES u, y ────────────────────────────────────────────────────
u = {}
for (p, a, e, d, t) in DSA_M:
    if (p, a, d, t) not in u:
        u[p, a, d, t] = model.addVar(vtype=GRB.BINARY, name=f"u_{p}_{a}_{d}_{t}")
for (p, a, d, t) in DSA_NM:
    if (p, a, d, t) not in u:
        u[p, a, d, t] = model.addVar(vtype=GRB.BINARY, name=f"u_{p}_{a}_{d}_{t}")

y = {(p,d,t): model.addVar(vtype=GRB.BINARY, name=f"y_{p}_{d}_{t}")
     for p in P for d in DAYS for t in T[:-1]}

model.update()
print(f"Variables: x={len(x)}  u={len(u)}  y={len(y)}")

# ── FUNCIÓN OBJETIVO ─────────────────────────────────────────────────────
model.setObjective(
    - W1 * gp.quicksum(u.values())
    - W2 * gp.quicksum((1 + Palt[p]) * y[p,d,t] for (p,d,t) in y),
    GRB.MAXIMIZE
)

# ─────────────────────────────────────────────────────────────────
#  RESTRICCIONES  (numeradas igual que la formulación matemática)
# ─────────────────────────────────────────────────────────────────

# 8.1 · Cobertura de actividades — profesionales médicos
for (p, a, e, d, t) in DSA_M:
    model.addConstr(
        gp.quicksum(x[p,a,b,d,t] for b in B) + u[p,a,d,t] == DSA_M[p,a,e,d,t],
        name=f"c8_1_{p}_{a}_{d}_{t}")

# 8.2 · Cobertura de actividades — profesionales no médicos
for (p, a, d, t) in DSA_NM:
    model.addConstr(
        gp.quicksum(x[p,a,b,d,t] for b in B) + u[p,a,d,t] == DSA_NM[p,a,d,t],
        name=f"c8_2_{p}_{a}_{d}_{t}")

# 8.3 · Capacidad del box
for b in B:
    for d in DAYS:
        for t in T:
            terms = [x[p,a,b,d,t] for p in P for a in A if (p,a,b,d,t) in x]
            if terms:
                model.addConstr(gp.quicksum(terms) <= cap_b.get(b,1), name=f"c8_3_{b}_{d}_{t}")

# 8.4 · Ubicación única (un profesional en un solo box por bloque)
for p in P:
    for d in DAYS:
        for t in T:
            terms = [x[p,a,b,d,t] for a in A for b in B if (p,a,b,d,t) in x]
            if terms:
                model.addConstr(gp.quicksum(terms) <= 1, name=f"c8_4_{p}_{d}_{t}")

# 8.5 · Compatibilidad clínica de la actividad
#      x_{p,a,b,d,t} <= COMBA_{b,a}
for (p, a, b, d, t) in x:
    model.addConstr(x[p,a,b,d,t] <= COMBA.get((b, a), 0),
                    name=f"c8_5_{p}_{a}_{b}_{d}_{t}")

# 8.6 · Compatibilidad policlínico – profesional

    if Forz.get((p, b, d, t), 0) == 1:
        continue
    rhs = gp.quicksum(BoxPoli_param.get((b, rho), 0) * HabPoli.get((p, rho), 0)
                      for rho in POLI)
    model.addConstr(x[p,a,b,d,t] <= rhs, name=f"c8_6_{p}_{a}_{b}_{d}_{t}")

# 8.7 · Registro de cambio de box entre bloques consecutivos
for p in P:
    for b in B:
        for d in DAYS:
            for t in T[:-1]:
                t1 = [x[p,a,b,d,t]    for a in A if (p,a,b,d,t)   in x]
                t2 = [x[p,a,bp,d,t+1] for a in A for bp in B if bp!=b and (p,a,bp,d,t+1) in x]
                if t1 or t2:
                    model.addConstr(y[p,d,t] >= gp.quicksum(t1)+gp.quicksum(t2)-1,
                                    name=f"c8_7_{p}_{b}_{d}_{t}")

# 8.8 · Bloqueo de asignaciones a boxes no habilitados
#      x_{p,a,b,d,t} <= HabBox_{p,b}·Σ_ρ BoxPoli_{b,ρ}
#  (se OMITE en slots forzados: allí manda 8.11, que puede habilitar
#   deliberadamente un box que HabBox no permitiría)
for (p, a, b, d, t) in x:
    if Forz.get((p, b, d, t), 0) == 1:
        continue
    rhs = HabBox.get((p, b), 0) * sum(BoxPoli_param.get((b, rho), 0) for rho in POLI)
    model.addConstr(x[p,a,b,d,t] <= rhs, name=f"c8_8_{p}_{a}_{b}_{d}_{t}")

# 8.9 · Compatibilidad especialidad médica – box  (solo p ∈ P_M)
#      x_{p,a,b,d,t} <= Σ_ρ BoxPoli_{b,ρ}·EspPoli_{e,ρ}·EspP_{p,e}
#  (con e = especialidad del profesional; se omite en slots forzados)
for (p, a, b, d, t) in x:
    prof = pid_to_prof[p]
    if prof["is_nm"]:
        continue
    if Forz.get((p, b, d, t), 0) == 1:
        continue
    e = prof["specialty"]
    rhs = gp.quicksum(
        BoxPoli_param.get((b, rho), 0) * EspPoli_param.get((e, rho), 0) * EspP.get((p, e), 0)
        for rho in POLI)
    model.addConstr(x[p,a,b,d,t] <= rhs, name=f"c8_9_{p}_{a}_{b}_{d}_{t}")

# 8.11 · Box forzado (REGLA ESPECIAL)
#        x_{p,a,b,d,t} = DSA_{p,a,d,t}   si Forz_{p,b,d,t}=1
r_applied = {r["id"]: 0 for r in REGLAS}
n_811 = 0
for (p, a, b, d, t) in x:
    if Forz.get((p, b, d, t), 0) == 1:
        dem = demanda(p, a, d, t)
        model.addConstr(x[p,a,b,d,t] == dem, name=f"c8_11_{p}_{a}_{b}_{d}_{t}")
        if dem == 1 and (p, a, d, t) in u:
            model.addConstr(u[p,a,d,t] == 0, name=f"c8_11u_{p}_{a}_{d}_{t}")
            for r in regla_pid_map:
                if r["pid"] == p and r["box"] == b and (r["dia"] == d or r["dia"] == "*") and t in r["blocks"]:
                    r_applied[r["id"]] += 1
        n_811 += 1

# 8.12 · Box exclusivo (REGLA ESPECIAL)
#        x_{p,a,b,d,t} <= Forz_{p,b,d,t}   ∀ b ∈ B^EX

n_812 = 0
for (p, a, b, d, t) in x:
    if b in B_EX:
        model.addConstr(x[p,a,b,d,t] <= Forz.get((p, b, d, t), 0),
                        name=f"c8_12_{p}_{a}_{b}_{d}_{t}")
        n_812 += 1


for rid, cnt in r_applied.items():
    print(f"  Regla {rid}: {cnt} restricciones (8.11) aplicadas")
print(f"  Restricciones 8.11 (box forzado)   : {n_811}")
print(f"  Restricciones 8.12 (box exclusivo) : {n_812}")
print(f"  Boxes exclusivos (B^EX)            : {sorted(B_EX)}")
print("Restricciones añadidas correctamente (8.1 a 8.12).")


# ============================================================
# SECCIÓN 5 – RESOLUCIÓN
# ============================================================

print("\n" + "=" * 60)
print("OPTIMIZANDO...")
print("=" * 60)
model.optimize()


# ============================================================
# SECCIÓN 6 – REPORTE
# ============================================================

STATUS_NAMES = {
    GRB.OPTIMAL:    "ÓPTIMO",
    GRB.TIME_LIMIT: "LÍMITE DE TIEMPO (solución factible)",
    GRB.INFEASIBLE: "INFACTIBLE",
    GRB.UNBOUNDED:  "NO ACOTADO",
}
print("\n" + "=" * 60)
print(f"ESTADO: {STATUS_NAMES.get(model.status, f'Código {model.status}')}")
print("=" * 60)

if model.status in (GRB.OPTIMAL, GRB.TIME_LIMIT) and model.SolCount > 0:
    print(f"Z = {model.ObjVal:.2f}")

    assignments = [(d,p,a,b,t) for (p,a,b,d,t),v in x.items() if v.X > 0.5]
    assignments.sort(key=lambda r: (DAYS.index(r[0]), r[1], r[4]))

    print(f"\nASIGNACIONES ({len(assignments)} bloques):")
    print(f"{'DIA':<12} {'T':<4} {'PROFESIONAL':<32} {'ACT':<5} {'BOX':<14}")
    print("-" * 75)
    for d,p,a,b,t in assignments:
        pr = pid_to_prof[p]
        flag = " ★" if any(r["box"]==b and r["pid"]==p for r in regla_pid_map) else ""
        print(f"{d:<12} {block_to_time(t):<5} {pr['name']:<32} {a:<5} {b:<14}{flag}")

    unassigned = [(p,a,d,t) for (p,a,d,t),v in u.items() if v.X > 0.5]
    if unassigned:
        print(f"\nSIN BOX ({len(unassigned)} bloques):")
        for p,a,d,t in sorted(unassigned, key=lambda r:(DAYS.index(r[2]),r[0],r[3])):
            print(f"  {d:<12} {block_to_time(t)} {pid_to_prof[p]['name']:<32} {a}")
    else:
        print("\n✅ Todas las actividades tienen box asignado.")

    print("\n" + "=" * 60)
    print("RESUMEN POR PROFESIONAL")
    print("=" * 60)
    total_dsa = total_asig = total_sin = 0
    for p in professionals:
        pid     = p["id"]
        asig    = sum(1 for (pp,_,_,d,t),v in x.items() if pp==pid and v.X>0.5)
        sin_box = sum(1 for (pp,_,d,t),v  in u.items() if pp==pid and v.X>0.5)
        cambios = sum(1 for (pp,d,t),v    in y.items() if pp==pid and v.X>0.5)
        tot     = (sum(1 for (pp,_,d,t) in DSA_NM if pp==pid) if p["is_nm"]
                   else sum(1 for (pp,_,_,d,t) in DSA_M if pp==pid))
        pct     = 100*asig/tot if tot>0 else 0
        total_dsa+=tot; total_asig+=asig; total_sin+=sin_box
        tipo = "NM" if p["is_nm"] else "MD"
        print(f"  [{tipo}] {p['name']:<32} {p['specialty']:<28} "
              f"DSA:{tot:3d} Asig:{asig:3d}({pct:5.1f}%) Sin:{sin_box:2d} CambBox:{cambios}")

    pct_g = 100*total_asig/total_dsa if total_dsa>0 else 0
    print(f"\n  TOTAL: DSA={total_dsa} Asig={total_asig}({pct_g:.1f}%) Sin={total_sin}")

    from collections import defaultdict as _dd
    ocup = _dd(list)
    for (p,a,b,d,t),v in x.items():
        if v.X>0.5: ocup[(d,t,b)].append(p)
    conflictos = {k:v for k,v in ocup.items() if len(v)>cap_b.get(k[2],1)}
    print(f"\n{'✅ Sin conflictos de capacidad.' if not conflictos else f'❌ {len(conflictos)} conflictos de capacidad.'}")

    box_ch = sum(1 for v in y.values() if v.X>0.5)
    pen_u  = W1*len(unassigned)
    pen_y  = sum(W2*(1+Palt[p])*v.X for (p,d,t),v in y.items())
    print(f"\nZ = -{pen_u:.0f}(u) - {pen_y:.0f}(y) = {-pen_u-pen_y:.1f}")
    print(f"Z Gurobi: {model.ObjVal:.1f}")

elif model.status == GRB.INFEASIBLE:
    print("Modelo infactible.")
    model.computeIIS()
    model.write("model_iis.ilp")
else:
    print("Sin solución.")


# ============================================================
# SECCIÓN 7 – EXPORTAR RESULTADOS AL EXCEL DE AGENDAS
# ============================================================

def export_to_agendas(input_path, output_path, x_vars, professionals_list, maestro):
    import shutil
    shutil.copy2(input_path, output_path)

    solution = {}
    if model.SolCount > 0:
        for (p,a,b,d,t),v in x_vars.items():
            if v.X > 0.5:
                solution[(p,a,d,t)] = b

    name_to_id = {normalize(pr["name"]): pr["id"] for pr in professionals_list}
    BOX_LABELS = maestro["labels"]

    HEADER_FILL  = PatternFill("solid", fgColor="1F4E79")
    HEADER_FONT  = Font(bold=True, color="FFFFFF", size=10)
    ASIG_FILL    = PatternFill("solid", fgColor="E2EFDA")
    MISS_FILL    = PatternFill("solid", fgColor="FCE4D6")
    NM_FILL      = PatternFill("solid", fgColor="EBF3FB")
    ESP_FILL     = PatternFill("solid", fgColor="FFF2CC")
    CENTER       = Alignment(horizontal="center", vertical="center", wrap_text=True)
    THIN         = Border(left=Side(style="thin"), right=Side(style="thin"),
                          top=Side(style="thin"),  bottom=Side(style="thin"))

    wb = openpyxl.load_workbook(output_path, keep_links=False)

    for sheet_name in wb.sheetnames:
        ws  = wb[sheet_name]
        pid = None
        is_nm_sh = False

        for ri in range(1, 20):
            cv = str(ws.cell(row=ri, column=1).value or "")
            if "NOMBRE PROFESIONAL:" in cv:
                rn = normalize(cv.replace("NOMBRE PROFESIONAL:","").strip())
                for sn, spid in name_to_id.items():
                    if sn in rn or rn in sn:
                        pid = spid; break
            if "PROFESION:" in cv.upper():
                pn = normalize(cv.upper().replace("PROFESION:","").strip())
                is_nm_sh = detect_nm_canon(pn, maestro["kw_to_canon"]) is not None

        if not pid:
            continue

        hdr_row = None
        for ri in range(1, ws.max_row+1):
            v0 = str(ws.cell(row=ri, column=1).value or "")
            v1 = str(ws.cell(row=ri, column=2).value or "")
            if "DIA" in v0 and "ACTIVIDAD" in v1:
                hdr_row = ri; break
        if not hdr_row:
            continue

        from openpyxl.utils import get_column_letter
        NC = ws.max_column + 1
        hc = ws.cell(row=hdr_row, column=NC, value="BOX ASIGNADO")
        hc.fill = HEADER_FILL; hc.font = HEADER_FONT
        hc.alignment = CENTER; hc.border = THIN
        ws.cell(row=hdr_row+1, column=NC).fill = HEADER_FILL
        ws.column_dimensions[get_column_letter(NC)].width = 28

        esp_boxes = {r["box"] for r in regla_pid_map if r["pid"]==pid}

        current_day = None
        for ri in range(hdr_row+2, ws.max_row+1):
            v_dia  = str(ws.cell(row=ri, column=1).value or "").strip()
            v_act  = str(ws.cell(row=ri, column=2).value or "").strip()
            v_tipo = str(ws.cell(row=ri, column=3).value or "").strip()
            d_raw  = ws.cell(row=ri, column=4).value
            h_raw  = ws.cell(row=ri, column=5).value
            v_d    = str(d_raw).strip() if d_raw is not None else "nan"
            v_h    = str(h_raw).strip() if h_raw is not None else "nan"

            if v_dia and v_dia.upper() in [d.upper() for d in DAYS]:
                current_day = v_dia.upper()
            if "NOMBRE" in v_act.upper() or "FIRMA" in v_act.upper():
                break
            if not v_act or v_act in ("nan","NaN") or not v_tipo or v_tipo in ("nan","NaN"):
                continue
            if current_day not in DAYS:
                continue
            if v_d in ("nan","NaN","None","") or v_h in ("nan","NaN","None",""):
                continue

            tu = normalize(v_tipo)
            if "SIN EXAMEN" in tu:      ac = "CSF"
            elif "EXAMEN" in tu:        ac = "CEF"
            elif "PROCEDIMIENTO" in tu: ac = "PROC"
            else: continue

            try:
                bf = time_to_block(d_raw); bt = time_to_block(h_raw)
            except Exception:
                continue

            box_counts = {}; missing = 0
            for t in range(bf, bt):
                if t not in range(1,(17*60-8*60)//30+1): continue
                bx = solution.get((pid, ac, current_day, t))
                if bx: box_counts[bx] = box_counts.get(bx,0)+1
                else:  missing += 1

            dc = ws.cell(row=ri, column=NC)
            dc.alignment = CENTER; dc.border = THIN

            if box_counts and missing == 0:
                dom = max(box_counts, key=box_counts.get)
                dc.value = BOX_LABELS.get(dom, dom)
                is_esp   = dom in esp_boxes
                dc.fill  = ESP_FILL if is_esp else (NM_FILL if is_nm_sh else ASIG_FILL)
                dc.font  = Font(bold=True, size=9,
                                color="7F6000" if is_esp else ("1F4E79" if is_nm_sh else "375623"))
                if len(box_counts) > 1:
                    alts = ", ".join(BOX_LABELS.get(b,b) for b in sorted(box_counts) if b!=dom)
                    dc.comment = xl_comments.Comment(
                        f"Cambio:\n  Principal: {BOX_LABELS.get(dom,dom)}\n  Alt: {alts}",
                        author="Modelo v6")
            else:
                dc.value = "Sin box asignado"
                dc.fill  = MISS_FILL
                dc.font  = Font(italic=True, color="9C0006", size=9)

    wb.save(output_path)
    print(f"Excel agendas guardado: '{output_path}'")


# ============================================================
# SECCIÓN 8 – EXPORTAR GRILLA DE DISTRIBUCIÓN
# ============================================================

def export_grilla(output_path, x_vars, professionals_list, maestro):
    from openpyxl.utils import get_column_letter

    def short_name(n):
        parts = n.strip().split()
        return " ".join(parts[:2]) if len(parts) >= 2 else n

    bdb = {}
    if model.SolCount > 0:
        for (p,a,b,d,t),v in x_vars.items():
            if v.X > 0.5:
                pr  = pid_to_prof[p]
                sn  = short_name(pr["name"]) + (" (NM)" if pr["is_nm"] else "")
                key = (b,d,t)
                bdb[key] = bdb[key]+"/"+sn if key in bdb and sn not in bdb[key] else sn

    BOX_LABELS = maestro["labels"]
    boxes_esp  = list(BOXES_EXCLUSIVOS_REGLA)
    boxes_nm   = list(BOXES_EXCLUSIVOS_NM)
    boxes_rest = [b for b in B if b not in boxes_nm and b not in boxes_esp]
    ALL_BOXES  = boxes_rest + boxes_nm + boxes_esp

    SEP_AFTER  = {boxes_rest[-1], boxes_nm[-1]} if boxes_nm else {boxes_rest[-1]}

    TF=PatternFill("solid",fgColor="1F4E79"); TS=Font(bold=True,color="FFFFFF",size=11)
    DF=PatternFill("solid",fgColor="2E75B6"); DS=Font(bold=True,color="FFFFFF",size=11)
    HF=PatternFill("solid",fgColor="BDD7EE"); HS=Font(bold=True,color="1F4E79",size=9)
    NF=PatternFill("solid",fgColor="D6E4F0")
    EF=PatternFill("solid",fgColor="FFF2CC"); ES=Font(bold=True,color="7F6000",size=9)
    SF=PatternFill("solid",fgColor="DEEAF1"); SS=Font(bold=True,color="1F4E79",size=8)
    BF=PatternFill("solid",fgColor="EBF5FB"); BS=Font(color="2C3E50",size=8)
    AF=PatternFill("solid",fgColor="E2EFDA"); NMF=PatternFill("solid",fgColor="EBF3FB")
    ESPF=PatternFill("solid",fgColor="FFF2CC")
    EMPT=PatternFill("solid",fgColor="F2F2F2")
    CA=Alignment(horizontal="center",vertical="center",wrap_text=True)
    LA=Alignment(horizontal="left",  vertical="center",wrap_text=True)
    TH=Border(left=Side(style="thin"),right=Side(style="thin"),
              top=Side(style="thin"), bottom=Side(style="thin"))
    MR=Border(left=Side(style="thin"),right=Side(style="medium"),
              top=Side(style="thin"), bottom=Side(style="thin"))
    TT=Border(left=Side(style="thin"),right=Side(style="thin"),
              top=Side(style="medium"),bottom=Side(style="thin"))
    TTMR=Border(left=Side(style="thin"),right=Side(style="medium"),
                top=Side(style="medium"),bottom=Side(style="thin"))

    wb  = openpyxl.Workbook()
    ws  = wb.active
    ws.title = "Distribucion Boxes"
    ws.column_dimensions["A"].width = 13

    col_idx = 2
    box_col = {}
    for b in ALL_BOXES:
        ws.column_dimensions[get_column_letter(col_idx)].width = 18
        box_col[b] = col_idx; col_idx += 1
        if b in SEP_AFTER:
            ws.column_dimensions[get_column_letter(col_idx)].width = 2
            col_idx += 1

    total_cols = col_idx - 1
    cur = 1

    ws.merge_cells(start_row=cur,start_column=1,end_row=cur,end_column=total_cols)
    tc=ws.cell(row=cur,column=1,value="DISTRIBUCIÓN DE POLICLÍNICOS — MODELO GUROBI v6")
    tc.fill=TF; tc.font=TS; tc.alignment=CA; cur+=1

    for day in DAYS:
        cur+=1
        ws.merge_cells(start_row=cur,start_column=1,end_row=cur,end_column=total_cols)
        dc=ws.cell(row=cur,column=1,value=day)
        dc.fill=DF; dc.font=DS; dc.alignment=CA; cur+=1

        ws.cell(row=cur,column=1,value="HORARIO").fill=HF
        ws.cell(row=cur,column=1).font=HS; ws.cell(row=cur,column=1).alignment=CA
        ws.cell(row=cur,column=1).border=TH
        for b in ALL_BOXES:
            col=box_col[b]
            is_nm_b  = b in boxes_nm
            is_esp_b = b in boxes_esp
            hfill = EF if is_esp_b else (NF if is_nm_b else HF)
            hfont = ES if is_esp_b else HS
            is_br = b in SEP_AFTER
            c=ws.cell(row=cur,column=col,value=BOX_LABELS.get(b,b))
            c.fill=hfill; c.font=hfont; c.alignment=CA
            c.border=TTMR if is_br else TH
        cur+=1

        prev_h=None
        for t in T:
            ts=block_to_time(t); te=block_to_time(t+1) if t<T[-1] else "17:00"
            lbl=f"{ts} - {te}"
            ch=int(ts.split(":")[0]); new_h=(ch!=prev_h); prev_h=ch

            hc=ws.cell(row=cur,column=1,value=lbl)
            hc.fill=SF if new_h else BF; hc.font=SS if new_h else BS
            hc.border=TT if new_h else TH; hc.alignment=CA

            for b in ALL_BOXES:
                col=box_col[b]
                is_nm_b=b in boxes_nm; is_esp_b=b in boxes_esp
                entry=bdb.get((b,day,t),"")
                is_br=b in SEP_AFTER
                brd=TTMR if (new_h and is_br) else (TT if new_h else (MR if is_br else TH))
                c=ws.cell(row=cur,column=col,value=entry)
                if entry:
                    c.fill=ESPF if is_esp_b else (NMF if is_nm_b else AF)
                    c.font=Font(color="7F6000" if is_esp_b else ("1F4E79" if is_nm_b else "375623"),size=8)
                else:
                    c.fill=EMPT; c.font=Font(size=8)
                c.alignment=LA; c.border=brd

            ws.row_dimensions[cur].height=18; cur+=1

    ws.freeze_panes="B3"
    wb.save(output_path)
    print(f"Grilla guardada: '{output_path}'")


if model.SolCount > 0:
    print("\n" + "=" * 60)
    print("EXPORTANDO RESULTADOS")
    print("=" * 60)
    out_agendas = str(Path(EXCEL_AGENDAS).with_name(f"{Path(EXCEL_AGENDAS).stem}_BoxAsignado.xlsx"))
    out_grilla  = str(Path(EXCEL_AGENDAS).with_name(f"Distribucion_Boxes_{Path(EXCEL_AGENDAS).stem.replace('Agendas_','')}.xlsx"))
    export_to_agendas(EXCEL_AGENDAS, out_agendas, x, professionals, M)
    export_grilla(out_grilla, x, professionals, M)
else:
    print("\nSin solución disponible.")

Agendas : Agendas_Hospital_Grant.xlsx
Maestro : Excel_Maestro_Boxes.xlsx

CARGANDO EXCEL MAESTRO

Maestro cargado:
  Boxes          : 16
  Policlínicos   : 1  → ['CIRUGIA INFANTIL']
  Especialidades : 4 → ['CIRUGIA INFANTIL Y ORTOPEDIA', 'CIRUJANO INFANTIL', 'CIRUJANO MAXILO-FACIAL', 'TRAUMATOLOGO INFANTIL']
  Profesiones NM : 4 → ['ENFERMERA', 'FONOAUDIOLOGO', 'ODONTOLOGO', 'PSICOLOGO']
  Reglas especiales: 4

Policlínico detectado desde nombre de archivo: 'CIRUGIA INFANTIL'

LEYENDO AGENDAS

Delta = 30 min  |T| = 18 bloques/día
Especialidades detectadas: ['TRAUMATOLOGO INFANTIL', 'CIRUJANO MAXILO-FACIAL', 'CIRUGIA INFANTIL Y ORTOPEDIA', 'CIRUJANO INFANTIL']

P_M  (25 médicos):
  P1_PROFESIONAL_1               | CIRUJANO INFANTIL              | poli=CIRUGIA INFANTIL palt=1
  P2_PROFESIONAL_2               | TRAUMATOLOGO INFANTIL          | poli=CIRUGIA INFANTIL palt=0
  P3_PROFESIONAL_3               | CIRUJANO INFANTIL              | poli=CIRUGIA INFANTIL palt=0
  P4_PROFESIONAL_4   

In [ ]:
model.update()  # por si acaso, para refrescar el estado interno

print(f"Variables totales      : {model.NumVars}  (bin={model.NumBinVars})")
print(f"Restricciones totales  : {model.NumConstrs}")
print(f"No-ceros en la matriz  : {model.NumNZs}")